# 03 · Download and check the data for the prediction game

This notebook downloads two extra datasets and checks them. Files that are already on disk are skipped,
so it is safe to re-run.

| Data | Source | Seasons |
|---|---|---|
| Match results (same columns as the existing `season-*.csv` files) | [football-data.co.uk](https://www.football-data.co.uk/englandm.php) | 2016-17 to 2020-21 (2021-22 onwards were already here) |
| Per-gameweek FPL player stats (points, minutes, goals, assists, price ...) | [vaastav/Fantasy-Premier-League](https://github.com/vaastav/Fantasy-Premier-League) | 2016-17 to 2025-26 |

Data fixes are done in `src/pl.py` (`load_fpl_season`):
- **2019-20**: the COVID restart is numbered GW 39-47 in FPL, so it is renumbered to 30-38; a postponed match also appears twice and only the real one is kept.
- **2019-20 restart and 2020-21**: matches had no crowd, so they get `No_Crowd = 1` and Elo gives no home advantage for them.
- **2016-17 to 2019-20**: the FPL files have no team/position columns, so they are rebuilt from `players_raw.csv` and the fixtures.
- **2024-25**: the "assistant manager" chip players are removed.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))   # run from notebooks/, so "src" is one folder up
from src import pl

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 140)

In [2]:
pl.download_old_matches()
pl.download_fpl()

have      season-1617.csv
have      season-1718.csv
have      season-1819.csv
have      season-1920.csv
have      season-2021.csv
have      2016-17/merged_gw.csv
have      2016-17/players_raw.csv
have      2017-18/merged_gw.csv
have      2017-18/players_raw.csv
have      2018-19/merged_gw.csv
have      2018-19/players_raw.csv
have      2019-20/merged_gw.csv
have      2019-20/players_raw.csv
have      2020-21/merged_gw.csv
have      2020-21/players_raw.csv
have      2021-22/merged_gw.csv
have      2021-22/players_raw.csv
have      2022-23/merged_gw.csv
have      2022-23/players_raw.csv
have      2023-24/merged_gw.csv
have      2023-24/players_raw.csv
have      2024-25/merged_gw.csv
have      2024-25/players_raw.csv
have      2025-26/merged_gw.csv
have      2025-26/players_raw.csv


## 1. Match results: the same checks as notebook 01

In [3]:
mdf, team_games, elo_hist = pl.build_match_table()

print("matches per season:", mdf.groupby("Season").size().to_dict())
print("duplicates:", mdf.duplicated(["Date", "Home_Team", "Away_Team"]).sum())
print("missing values in raw columns:", mdf[["Home_Goals", "Away_Goals", "Home_Shots", "Away_Shots"]].isna().sum().sum())

expected = np.select([mdf["Home_Goals"] > mdf["Away_Goals"], mdf["Home_Goals"] < mdf["Away_Goals"]], ["H", "A"], "D")
print("FTR consistent:", (expected == mdf["Full_Time_Result"]).all())
print("No-crowd matches:", mdf["No_Crowd"].sum())

assert (mdf.groupby("Season").size() == 380).all()
assert (expected == mdf["Full_Time_Result"]).all()

matches per season: {'2016-17': 380, '2017-18': 380, '2018-19': 380, '2019-20': 380, '2020-21': 380, '2021-22': 380, '2022-23': 380, '2023-24': 380, '2024-25': 380, '2025-26': 380}
duplicates: 0
missing values in raw columns: 0
FTR consistent: True
No-crowd matches: 472


## 2. FPL gameweek data

In [4]:
fpl = pl.load_fpl()
fixtures = pl.fpl_fixtures(fpl)

summary = fpl.groupby("Season").agg(rows=("Points", "size"), players=("Player_ID", "nunique"),
                                    gameweeks=("GW", "nunique"), fixtures=("Fixture", "nunique"))
print("missing team names:", fpl["Team"].isna().sum() + fpl["Opponent"].isna().sum())
summary

missing team names: 0


,rows,players,gameweeks,fixtures
Season,,,,
2016-17,23679,683,38,380
2017-18,22467,647,38,380
2018-19,21790,624,38,380
2019-20,22501,666,38,380
2020-21,24365,713,38,380
2021-22,25447,737,38,380
2022-23,26505,778,37,380
2023-24,29725,865,38,380
2024-25,27283,784,38,380


## 3. Do the two sources agree?

Every FPL fixture should be found in the match CSVs with the same score, and each season's player goals
plus own goals should equal the total goals in the match CSVs.

In [5]:
check = fixtures.merge(mdf[["Season", "Home_Team", "Away_Team", "Home_Goals", "Away_Goals"]],
                       on=["Season", "Home_Team", "Away_Team"], how="left", suffixes=("", "_csv"))
not_found = check["Home_Goals_csv"].isna().sum()
wrong = ((check["Home_Goals"] != check["Home_Goals_csv"]) | (check["Away_Goals"] != check["Away_Goals_csv"])).sum()
print("FPL fixtures not found in match CSVs:", not_found, "| score mismatches:", wrong)

goals = pd.DataFrame({
    "FPL goals + own goals": fpl.groupby("Season")[["Goals", "Own_Goals"]].sum().sum(axis=1),
    "Match CSV goals": mdf.groupby("Season")[["Home_Goals", "Away_Goals"]].sum().sum(axis=1),
})
goals["Difference"] = goals.iloc[:, 0] - goals.iloc[:, 1]
print(goals)

assert not_found == 0 and wrong == 0
assert fpl["Team"].notna().all() and fpl["Opponent"].notna().all()
assert (fixtures.groupby("Season").size() == 380).all()
assert fixtures["GW"].between(1, 38).all()

FPL fixtures not found in match CSVs: 0 | score mismatches: 0
         FPL goals + own goals  Match CSV goals  Difference
Season                                                     
2016-17                   1064             1064           0
2017-18                   1018             1018           0
2018-19                   1072             1072           0
2019-20                   1034             1034           0
2020-21                   1024             1024           0
2021-22                   1071             1071           0
2022-23                   1084             1084           0
2023-24                   1246             1246           0
2024-25                   1115             1115           0
2025-26                   1045             1045           0


Small goal differences can happen: FPL sometimes corrects a goal scorer late, or scores a player who
appeared in a match that was later changed. Anything more than a handful per season would be a bug.